In [ ]:
import pandas as pd

# Load CSV
df = pd.read_excel("GrideH9_v3.xlsx")

# Basic overview
print("Shape of dataset:", df.shape)
print("\nColumn names:\n", df.columns)

In [ ]:
# =========================================================
# Cell 1: Preprocessing + Split
# =========================================================
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# =========================
# 0. Load Data
# =========================
df_raw = pd.read_excel("GrideH9_v3.xlsx")
df = df_raw.copy()

# =========================
# 1. Define Target
# =========================
y_reg_raw = df["VolBoth_sum"].copy()

# =========================
# 2. Remove Leakage Features
# =========================
drop_cols = [
    "VolCD_avg","VolVG_avg","VolCD_sum","VolVG_sum","VolBoth_sum",
    "VolCD","VolVG",                                                # leakage: raw volume columns
    "Bin_CD","Bin_VG","Bin_Both",
    "GRID_ID",
    "age_sum","age_med","num_story_sum","num_story_med",
    "sqm","sqm_med",
    "founde_ht_sum","found_ht_med",
    "val_struct","val_struct_med",
    "val_cont","val_cont_med","landuse"
]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

# =========================
# 3. Feature Engineering
# dist_htrack -> use the smaller value
# windgust    -> use the larger value
# rainfall    -> use the larger value
# =========================
df["dist_htrack"] = df[["dist_htrack_M", "dist_htrack_H"]].min(axis=1)
df = df.drop(columns=["dist_htrack_M", "dist_htrack_H"], errors="ignore")

df["windgust"] = df[["windgust_M", "windgust_H"]].max(axis=1)
df = df.drop(columns=["windgust_M", "windgust_H"], errors="ignore")

df["rainfall"] = df[["rainfall_M", "rainfall_H"]].max(axis=1)
df = df.drop(columns=["rainfall_M", "rainfall_H"], errors="ignore")

print("=== Feature Engineering ===")
print(f"dist_htrack: {df['dist_htrack'].describe().to_dict()}")
print(f"windgust:    {df['windgust'].describe().to_dict()}")
print(f"rainfall:    {df['rainfall'].describe().to_dict()}")

# =========================
# 4. X Outlier Clipping
# =========================
log_cols  = ["sqm_sum", "val_struct_sum", "val_cont_sum", "fld_pct"]
cont_cols = df.select_dtypes(include=["int64","float64"]).columns.tolist()
cont_cols = [c for c in cont_cols if not c.startswith(("occtype","bldg"))]
cont_cols = [c for c in cont_cols if c not in ["dist_coast","dist_reservoir"]]
cont_cols = [c for c in cont_cols if c not in log_cols]

clip_summary = []
for col in cont_cols:
    skew = df[col].skew()
    if skew > 3:    upper = df[col].quantile(0.80)
    elif skew > 2:  upper = df[col].quantile(0.90)
    elif skew > 1:  upper = df[col].quantile(0.98)
    else:           upper = df[col].quantile(0.995)
    n_clipped = (df[col] > upper).sum()
    clip_summary.append({
        "feature": col, "skew": round(skew, 2),
        "upper": round(upper, 2), "n_clipped": n_clipped,
        "percent_clipped": round(n_clipped / len(df[col]) * 100, 2)
    })
    df[col] = np.clip(df[col], 0, upper)

clip_df = pd.DataFrame(clip_summary).sort_values(by="percent_clipped", ascending=False)
print("\n===== X Clipping Summary =====")
print(clip_df)

# =========================
# 5. Log Transform
# =========================
for col in log_cols:
    if col in df.columns:
        df[col] = np.log1p(df[col])

# =========================
# 6. Binary Distance Features
# =========================
df["near_coast"]     = (df_raw["dist_coast"]     < 3).astype(int)
df["near_reservoir"] = (df_raw["dist_reservoir"] < 5).astype(int)
df = df.drop(columns=["dist_coast","dist_reservoir"], errors="ignore")

print("\n=== near_coast ===")
print(f"Close (1): {(df['near_coast']==1).sum()}")
print(f"Far   (0): {(df['near_coast']==0).sum()}")
print("\n=== near_reservoir ===")
print(f"Close (1): {(df['near_reservoir']==1).sum()}")
print(f"Far   (0): {(df['near_reservoir']==0).sum()}")

# =========================
# 8. Ordinal Encoding
# =========================
evac_map = {"none": 0, "low": 1, "med": 2, "high": 3}
df["evac_degree"] = df["evac_degree"].map(evac_map)
print("\n=== evac_degree (Ordinal) ===")
print("  0=none  1=low  2=med  3=high")
vc = df["evac_degree"].value_counts().sort_index()
label_map_evac = {0: "none", 1: "low", 2: "med", 3: "high"}
for k, v in vc.items():
    print(f"  {k} ({label_map_evac[k]:>4}): {v}")

fld_map = {"X": 0, "A": 1, "AO": 1, "AE": 2, "VE": 3}
df["fld_zone"] = df["fld_zone"].map(fld_map)
print("\n=== fld_zone (Ordinal) ===")
print("  0=X(low risk)  1=A/AO  2=AE  3=VE(high risk)")
vc = df["fld_zone"].value_counts().sort_index()
label_map_fld = {0: "X", 1: "A/AO", 2: "AE", 3: "VE"}
for k, v in vc.items():
    print(f"  {k} ({label_map_fld[k]:>4}): {v}")

# =========================
# 9. Dummy Encoding
# =========================
df_model = pd.get_dummies(df.copy(), drop_first=False)
print(f"\n=== df_model shape: {df_model.shape} ===")
print(f"Columns: {df_model.columns.tolist()}")

# =========================
# 10. Automate split processing for each clipping level
# =========================
clip_percentiles = [50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 99]
y_pos_raw  = y_reg_raw[y_reg_raw > 0]
clip_results = {}

fig1, axes1 = plt.subplots(2, 6, figsize=(36, 12))
fig2, axes2 = plt.subplots(2, 6, figsize=(36, 12))
axes1 = axes1.flatten()
axes2 = axes2.flatten()

for i, cp in enumerate(clip_percentiles):

    y_upper   = y_pos_raw.quantile(cp / 100)
    n_clipped = (y_pos_raw > y_upper).sum()
    y_reg_cp  = y_reg_raw.clip(upper=y_upper)
    y_pos_cp  = y_reg_cp[y_reg_cp > 0]

    median_val          = int(y_pos_cp.median())
    thr_candidates_init = list(range(max(50, median_val - 200), median_val + 225, 25))

    thr_results = []
    for thr in thr_candidates_init:
        n_low  = (y_pos_cp <= thr).sum()
        n_high = (y_pos_cp >  thr).sum()
        ratio  = n_low / (n_low + n_high)
        thr_results.append({
            "Threshold": thr,
            "Low_ratio": round(ratio, 3),
            "Balance":   round(abs(ratio - 0.5), 3)
        })

    thr_df_init = pd.DataFrame(thr_results)
    best_thr    = int(thr_df_init.loc[thr_df_init["Balance"].idxmin(), "Threshold"])

    y_stratify = y_reg_cp.apply(
        lambda x: "zero" if x == 0 else ("low" if x <= best_thr else "high")
    )

    X_train_cp, X_test_cp, _, _ = train_test_split(
        df_model, y_stratify,
        test_size=0.15,
        random_state=42,
        stratify=y_stratify
    )

    y_train_cp = y_reg_cp.loc[X_train_cp.index]
    y_test_cp  = y_reg_cp.loc[X_test_cp.index]

    train_zero = (y_train_cp == 0).sum()
    train_low  = ((y_train_cp > 0) & (y_train_cp <= best_thr)).sum()
    train_high = (y_train_cp > best_thr).sum()
    test_zero  = (y_test_cp  == 0).sum()
    test_low   = ((y_test_cp  > 0) & (y_test_cp  <= best_thr)).sum()
    test_high  = (y_test_cp  > best_thr).sum()

    clip_results[cp] = {
        "y_reg":       y_reg_cp,
        "y_upper":     y_upper,
        "n_clipped":   n_clipped,
        "threshold":   best_thr,
        "X_train":     X_train_cp,
        "X_test":      X_test_cp,
        "y_train_reg": y_train_cp,
        "y_test_reg":  y_test_cp,
    }

    print(f"\nClip={cp}%  upper={y_upper:.2f}  n_clipped={n_clipped}  best_thr={best_thr}")
    print(f"Train: {len(X_train_cp)}  Zero={train_zero}  Low={train_low}  High={train_high}  Low_ratio={train_low/(train_low+train_high):.3f}")
    print(f"Test : {len(X_test_cp)}   Zero={test_zero}   Low={test_low}   High={test_high}   Low_ratio={test_low/(test_low+test_high):.3f}")

    # Histogram
    axes1[i].hist(y_pos_cp, bins=50, color="steelblue", edgecolor="white")
    axes1[i].axvline(best_thr,    color="red",   linestyle="--", lw=1.5, label=f"Best={best_thr}")
    axes1[i].axvline(median_val,  color="green", linestyle="--", lw=1.5, label=f"Median={median_val}")
    axes1[i].set_title(f"Clip={cp}%  upper={y_upper:.0f}\nn_clipped={n_clipped}")
    axes1[i].set_xlabel("VolBoth_sum")
    axes1[i].set_ylabel("Count")
    axes1[i].legend(fontsize=7)
    axes1[i].grid(alpha=0.3)

    # Train/Test Distribution
    x = np.arange(3)
    tr_vals = [train_zero, train_low, train_high]
    te_vals = [test_zero,  test_low,  test_high]
    axes2[i].bar(x - 0.2, tr_vals, 0.4, color="steelblue", label="Train")
    axes2[i].bar(x + 0.2, te_vals, 0.4, color="tomato",    label="Test")
    axes2[i].set_xticks(x)
    axes2[i].set_xticklabels(["Zero", "Low", "High"])
    axes2[i].set_title(f"Distribution  Clip={cp}%")
    axes2[i].legend(fontsize=7)
    axes2[i].grid(alpha=0.3, axis="y")
    for j, (tr, te) in enumerate(zip(tr_vals, te_vals)):
        axes2[i].text(j - 0.2, tr + 3, str(tr), ha="center", fontsize=8)
        axes2[i].text(j + 0.2, te + 3, str(te), ha="center", fontsize=8)

for j in range(i + 1, len(axes1)):
    axes1[j].set_visible(False)
    axes2[j].set_visible(False)

fig1.suptitle("y Distribution per Clip", fontsize=13)
fig1.tight_layout()
fig1.show()

fig2.suptitle("Train/Test Distribution per Clip", fontsize=13)
fig2.tight_layout()
fig2.show()

print("\n[Cell 1 Done]")
print(f"{'Clip':>6} {'Upper':>8} {'N_clip':>8} {'Best_Thr':>10} {'Train':>7} {'Test':>7}")
print("-" * 50)
for cp in clip_percentiles:
    r = clip_results[cp]
    print(f"{cp:>6} {r['y_upper']:>8.2f} {r['n_clipped']:>8} "
          f"{r['threshold']:>10} {len(r['X_train']):>7} {len(r['X_test']):>7}")

In [ ]:
# =========================================================
# Cell 2-A: Layer 1 — Zero vs Positive Classification
# Fixed at clip=99% (classification result is clip-independent)
# =========================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_curve, auc
)
from imblearn.over_sampling import SMOTE

# =========================================================
# 0. Parameters
# =========================================================
RF_PARAMS_CLF = dict(
    n_estimators     = 50,
    max_depth        = 10,
    min_samples_leaf = 10,
    max_features     = 7,
    random_state     = 42,
    n_jobs           = -1
)

# =========================================================
# 1. Data Setup (fixed at clip=99%)
# =========================================================
cp          = 99
X_train     = clip_results[cp]["X_train"]
X_test      = clip_results[cp]["X_test"]
y_train_reg = clip_results[cp]["y_train_reg"]
y_test_reg  = clip_results[cp]["y_test_reg"]

y_train_binary = (y_train_reg > 0).astype(int)
y_test_binary  = (y_test_reg  > 0).astype(int)

print(f"Train — Zero: {(y_train_binary==0).sum()}  Positive: {(y_train_binary==1).sum()}")
print(f"Test  — Zero: {(y_test_binary==0).sum()}   Positive: {(y_test_binary==1).sum()}")

# =========================================================
# [LEAKAGE CHECK] Features perfectly correlated with target
# =========================================================
corr = X_train.corrwith(y_train_binary.astype(float)).abs()
top_corr = corr.sort_values(ascending=False).head(20)
print("\n=== Top 20 Feature–Target Correlations (LEAKAGE CHECK) ===")
print(top_corr.to_string())

# Drop any feature with |corr| > 0.95 (near-perfect predictor)
leakage_cols = corr[corr > 0.95].index.tolist()
if leakage_cols:
    print(f"\n[WARNING] Dropping {len(leakage_cols)} leakage features: {leakage_cols}")
    X_train = X_train.drop(columns=leakage_cols)
    X_test  = X_test.drop(columns=leakage_cols)
else:
    print("\n[OK] No near-perfect leakage features detected.")

bldg_cols    = [c for c in X_train.columns if c.startswith("bldg")]
occtype_cols = [c for c in X_train.columns if c.startswith("occtype")]
grouped_cols = set(bldg_cols + occtype_cols)

# =========================================================
# 2. Feature Importance → Top 20 Selection
#    Train on original (pre-SMOTE) data to get unbiased importances
# =========================================================
rf_feat = RandomForestClassifier(**RF_PARAMS_CLF)
rf_feat.fit(X_train, y_train_binary)

feat_imp_raw = pd.DataFrame({
    "feature":    X_train.columns,
    "importance": rf_feat.feature_importances_
}).sort_values("importance", ascending=False)

grouped_imp = {
    "bldg_type": feat_imp_raw[feat_imp_raw["feature"].isin(bldg_cols)]["importance"].sum(),
    "occtype":   feat_imp_raw[feat_imp_raw["feature"].isin(occtype_cols)]["importance"].sum(),
}
for _, row in feat_imp_raw.iterrows():
    if row["feature"] not in grouped_cols:
        grouped_imp[row["feature"]] = row["importance"]

grouped_imp_df = (
    pd.DataFrame(grouped_imp.items(), columns=["feature", "importance"])
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print("\n" + "=" * 60)
print("Feature Importance (Top 20 marked)")
print("=" * 60)
print(f"{'Rank':>4}  {'Feature':30} {'Importance':>12}  {'':>8}")
print("-" * 60)
for i, row in grouped_imp_df.iterrows():
    tag = "* TOP20" if i < 20 else ""
    print(f"{i+1:>4}  {row['feature']:30} {row['importance']:>12.6f}  {tag}")

def expand_features(group_list, bldg_cols, occtype_cols):
    expanded = []
    for g in group_list:
        if g == "bldg_type": expanded.extend(bldg_cols)
        elif g == "occtype":  expanded.extend(occtype_cols)
        else:                 expanded.append(g)
    return expanded

top_features = expand_features(
    grouped_imp_df.head(20)["feature"].tolist(), bldg_cols, occtype_cols
)
print(f"\nTop 20 groups → expanded to {len(top_features)} columns")

# =========================================================
# 3. SMOTE + Train
# =========================================================
smote_l1 = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote_l1.fit_resample(
    X_train[top_features], y_train_binary
)
print(f"\nAfter SMOTE — Zero: {(y_train_sm==0).sum()}  Positive: {(y_train_sm==1).sum()}")

clf_l1 = RandomForestClassifier(**RF_PARAMS_CLF)
clf_l1.fit(X_train_sm, y_train_sm)

# =========================================================
# 4. Evaluation
#    Train acc: evaluated on ORIGINAL (pre-SMOTE) train set
#    Test  acc: evaluated on held-out test set
# =========================================================
# Correct: use original X_train, not SMOTE-augmented X_train_sm
y_train_pred_orig = clf_l1.predict(X_train[top_features])
train_acc_l1      = (y_train_pred_orig == y_train_binary).mean()

y_pred_l1  = clf_l1.predict(X_test[top_features])
y_prob_l1  = clf_l1.predict_proba(X_test[top_features])[:, 1]

cm             = confusion_matrix(y_test_binary, y_pred_l1)
tn, fp, fn, tp = cm.ravel()
test_acc_l1    = (tn + tp) / (tn + fp + fn + tp)

fpr_l1, tpr_l1, _ = roc_curve(y_test_binary, y_prob_l1)
auc_l1             = auc(fpr_l1, tpr_l1)

report = classification_report(
    y_test_binary, y_pred_l1,
    target_names=["Zero", "Positive"],
    output_dict=True,
    zero_division=0
)

# =========================================================
# 5. Print Report
# =========================================================
print("\n" + "=" * 70)
print("LAYER 1 — Classification Report")
print("=" * 70)

report_df = pd.DataFrame({
    "Class":        ["Zero (No)", "Positive (Yes)", "Macro Avg"],
    "N (actual)":   [tn + fp,     fn + tp,          tn + fp + fn + tp],
    "Correct":      [tn,          tp,                tn + tp],
    "Wrong":        [fp,          fn,                fp + fn],
    "Accuracy(%)":  [round(tn/(tn+fp)*100, 2) if (tn+fp) > 0 else 0,
                     round(tp/(tp+fn)*100, 2) if (tp+fn) > 0 else 0,
                     round(test_acc_l1*100, 2)],
    "Precision(%)": [round(report["Zero"]["precision"]      * 100, 2),
                     round(report["Positive"]["precision"]  * 100, 2),
                     round(report["macro avg"]["precision"] * 100, 2)],
    "Recall(%)":    [round(report["Zero"]["recall"]         * 100, 2),
                     round(report["Positive"]["recall"]     * 100, 2),
                     round(report["macro avg"]["recall"]    * 100, 2)],
    "F1(%)":        [round(report["Zero"]["f1-score"]       * 100, 2),
                     round(report["Positive"]["f1-score"]   * 100, 2),
                     round(report["macro avg"]["f1-score"]  * 100, 2)],
})
print(report_df.to_string(index=False))
print(f"\n  Train Acc (original) : {train_acc_l1*100:.2f}%")
print(f"  Test  Acc            : {test_acc_l1*100:.2f}%")
print(f"  Gap                  : {(train_acc_l1 - test_acc_l1)*100:.2f}%")
print(f"  AUC                  : {auc_l1:.4f}")

# =========================================================
# 6. Plots — ROC + Confusion Matrix
# =========================================================
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

# ROC Curve
axes[0].plot(fpr_l1, tpr_l1, color="steelblue", lw=2, label=f"AUC = {auc_l1:.3f}")
axes[0].plot([0, 1], [0, 1], "k--", lw=1)
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title(
    f"Layer 1 — ROC Curve\n"
    f"Train={train_acc_l1*100:.2f}%  Test={test_acc_l1*100:.2f}%  "
    f"Gap={(train_acc_l1-test_acc_l1)*100:.2f}%"
)
axes[0].legend()
axes[0].grid(alpha=0.3)

# Confusion Matrix
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["No", "Yes"]
).plot(ax=axes[1], colorbar=False, cmap="Blues")
axes[1].set_title(
    f"Layer 1 — Confusion Matrix\n"
    f"Test Acc={test_acc_l1*100:.2f}%  AUC={auc_l1:.4f}"
)
axes[1].set_xlabel("Predicted label")
axes[1].set_ylabel("True label")

plt.tight_layout()
plt.show()

# =========================================================
# 7. Store Results per Clip
# =========================================================
train_pos_mask = y_train_reg > 0
test_pred_pos  = pd.Series(y_pred_l1, index=X_test.index) == 1

clf_results = {}
for cp_reg in [50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 99]:
    y_reg_cp = clip_results[cp_reg]["y_reg"]
    clf_results[cp_reg] = {
        "clf_l1":       clf_l1,
        "top_features": top_features,
        "bldg_cols":    bldg_cols,
        "occtype_cols": occtype_cols,
        "grouped_cols": grouped_cols,
        "auc_l1":       auc_l1,
        "train_acc":    train_acc_l1,
        "test_acc":     test_acc_l1,
        "gap":          train_acc_l1 - test_acc_l1,
        "X_train_pos":  X_train.loc[train_pos_mask],
        "y_train_pos":  y_reg_cp.loc[X_train.index].loc[train_pos_mask],
        "X_test_pos":   X_test.loc[test_pred_pos],
        "y_test_pos":   y_reg_cp.loc[X_test.index].loc[test_pred_pos],
        "f1_zero":      round(report["Zero"]["f1-score"]      * 100, 2),
        "f1_positive":  round(report["Positive"]["f1-score"]  * 100, 2),
        "f1_macro":     round(report["macro avg"]["f1-score"] * 100, 2),
    }

print(f"\n[Done] Train pos={train_pos_mask.sum()}  Test pos={test_pred_pos.sum()}")

In [ ]:
# =========================================================
# Cell 2-B: Layer 2 — Regression (Independent Model)
# Separate 85:15 split for y>0 samples only
# Clip x Threshold Sweep (Log Transform) + SMOGN
# =========================================================
import smogn
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

def calc_metrics(y_true, y_pred):
    rmse       = np.sqrt(mean_squared_error(y_true, y_pred))
    mae        = np.mean(np.abs(y_true - y_pred))
    nmae       = mae / (y_true.max() - y_true.min())
    r2         = r2_score(y_true, y_pred)
    nrmse_rng  = rmse / (y_true.max() - y_true.min()) * 100
    nrmse_std  = rmse / y_true.std() * 100
    cov        = rmse / y_true.mean()
    agg        = np.abs(y_true - y_pred).sum() / y_true.sum() * 100
    return rmse, mae, nmae, r2, nrmse_rng, nrmse_std, cov, agg

RF_PARAMS_REG = dict(
    n_estimators     = 50,
    max_depth        = None,
    min_samples_leaf = 1,
    max_features     = 7,
    random_state     = 42,
    n_jobs           = -1
)

clip_percentiles = [50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 99]
thr_candidates   = list(range(100, 1500, 25))
all_reg_results  = []

# =========================================================
# Common Columns
# =========================================================
bldg_cols    = [c for c in df_model.columns if c.startswith("bldg")]
occtype_cols = [c for c in df_model.columns if c.startswith("occtype")]
grouped_cols = set(bldg_cols + occtype_cols)

# =========================================================
# Separate 85:15 Split for y>0 Samples (Regression Only)
# =========================================================
pos_mask   = y_reg_raw > 0
df_pos     = df_model[pos_mask]
y_pos_raw2 = y_reg_raw[pos_mask]

print(f"Number of y>0 samples: {len(df_pos)}")

clip_results_pos = {}

for cp in clip_percentiles:
    y_upper  = clip_results[cp]["y_upper"]
    y_reg_cp = y_pos_raw2.clip(upper=y_upper)

    y_stratify_pos = y_reg_cp.apply(
        lambda x: "low" if x <= clip_results[cp]["threshold"] else "high"
    )

    X_train_pos, X_test_pos, _, _ = train_test_split(
        df_pos, y_stratify_pos,
        test_size=0.15,
        random_state=42,
        stratify=y_stratify_pos
    )

    y_train_pos = y_reg_cp.loc[X_train_pos.index]
    y_test_pos  = y_pos_raw2.loc[X_test_pos.index]

    clip_results_pos[cp] = {
        "X_train":     X_train_pos,
        "X_test":      X_test_pos,
        "y_train_reg": y_train_pos,
        "y_test_reg":  y_test_pos,
        "y_reg":       y_reg_cp,
        "y_upper":     y_upper,
        "threshold":   clip_results[cp]["threshold"],
    }

    print(f"Clip={cp}%  Train={len(X_train_pos)}  Test={len(X_test_pos)}")

# =========================================================
# Top 20 Features — Computed Once Based on cp=99%
# =========================================================
cp_ref      = 99
X_train_ref = clip_results_pos[cp_ref]["X_train"]
y_tr_ref    = clip_results_pos[cp_ref]["y_train_reg"]

reg_feat_tmp = RandomForestRegressor(**RF_PARAMS_REG)
reg_feat_tmp.fit(X_train_ref, np.log1p(y_tr_ref))

fi_reg_raw = pd.DataFrame({
    "Feature":    X_train_ref.columns,
    "Importance": reg_feat_tmp.feature_importances_
}).sort_values("Importance", ascending=False)

grouped_imp_reg = {}
grouped_imp_reg["bldg_type"] = fi_reg_raw[fi_reg_raw["Feature"].isin(bldg_cols)]["Importance"].sum()
grouped_imp_reg["occtype"]   = fi_reg_raw[fi_reg_raw["Feature"].isin(occtype_cols)]["Importance"].sum()
for _, row in fi_reg_raw.iterrows():
    if row["Feature"] not in grouped_cols:
        grouped_imp_reg[row["Feature"]] = row["Importance"]

grouped_imp_reg_df = pd.DataFrame(
    grouped_imp_reg.items(), columns=["feature", "importance"]
).sort_values("importance", ascending=False).reset_index(drop=True)

top20_reg_groups = grouped_imp_reg_df.head(20)["feature"].tolist()
top20_reg        = expand_features(top20_reg_groups, bldg_cols, occtype_cols)

print("\n--- Top 20 Features (Reference: cp=99%) ---")
print(f"{'Rank':>4}  {'Feature':30} {'Importance':>12}")
print("-" * 50)
for idx, row in grouped_imp_reg_df.iterrows():
    tag = "* TOP20" if idx < 20 else ""
    print(f"{idx+1:>4}  {row['feature']:30} {row['importance']:>12.6f}  {tag}")
print(f"\nTop 20 groups → expanded to {len(top20_reg)} columns")

# =========================================================
# Clip x Threshold Sweep
# =========================================================
print(f"\nThreshold candidates: {len(thr_candidates)}  range: {thr_candidates[0]}~{thr_candidates[-1]}")

for cp in clip_percentiles:

    print(f"\n{'='*60}")
    print(f"Clip={cp}%")
    print(f"{'='*60}")

    X_train_pos = clip_results_pos[cp]["X_train"]
    X_test_pos  = clip_results_pos[cp]["X_test"]
    y_tr_pos    = clip_results_pos[cp]["y_train_reg"]
    y_te_pos    = clip_results_pos[cp]["y_test_reg"]
    y_upper     = clip_results_pos[cp]["y_upper"]

    print(f"Train pos: {len(X_train_pos)}  Test pos: {len(X_test_pos)}")

    for thr in thr_candidates:

        if thr >= y_upper:
            continue

        low_mask_tr  = y_tr_pos <= thr
        high_mask_tr = y_tr_pos >  thr

        n_low_tr  = low_mask_tr.sum()
        n_high_tr = high_mask_tr.sum()

        if n_low_tr < 5 or n_high_tr < 5:
            continue

        low_ratio = n_low_tr / (n_low_tr + n_high_tr)
        if not (0.30 <= low_ratio <= 0.70):
            continue

        # Tier Classifier (SMOTE)
        tier_label_tr = pd.Series(
            np.where(y_tr_pos <= thr, "low", "high"),
            index=y_tr_pos.index
        )
        try:
            smote_l2 = SMOTE(random_state=42)
            X_sm_l2, y_sm_l2 = smote_l2.fit_resample(
                X_train_pos[top20_reg], tier_label_tr
            )
        except Exception:
            X_sm_l2 = X_train_pos[top20_reg]
            y_sm_l2 = tier_label_tr

        clf_tier = RandomForestClassifier(**RF_PARAMS_REG)
        clf_tier.fit(X_sm_l2, y_sm_l2)

        train_tier_acc = (clf_tier.predict(X_sm_l2) == y_sm_l2).mean()
        tier_pred      = clf_tier.predict(X_test_pos[top20_reg])
        tier_true      = np.where(y_te_pos <= thr, "low", "high")
        test_tier_acc  = (tier_pred == tier_true).mean()

        low_pred_te  = tier_pred == "low"
        high_pred_te = tier_pred == "high"

        # =========================================================
        # SMOGN for Low / High Regressors
        # =========================================================
        # Low Regressor
        try:
            data_low = X_train_pos[top20_reg][low_mask_tr].copy().reset_index(drop=True)
            data_low["target"] = np.log1p(y_tr_pos[low_mask_tr].values)
            data_low_sm = smogn.smoter(
                data=data_low,
                y="target",
                k=min(5, len(data_low) - 1),
                samp_method="balance",
                seed=42
            )
            X_low_sm = data_low_sm.drop(columns=["target"])
            y_low_sm = data_low_sm["target"]
        except Exception:
            X_low_sm = X_train_pos[top20_reg][low_mask_tr]
            y_low_sm = np.log1p(y_tr_pos[low_mask_tr])

        reg_low = RandomForestRegressor(**RF_PARAMS_REG)
        reg_low.fit(X_low_sm, y_low_sm)

        # High Regressor
        try:
            data_high = X_train_pos[top20_reg][high_mask_tr].copy().reset_index(drop=True)
            data_high["target"] = np.log1p(y_tr_pos[high_mask_tr].values)
            data_high_sm = smogn.smoter(
                data=data_high,
                y="target",
                k=min(5, len(data_high) - 1),
                samp_method="balance",
                seed=42
            )
            X_high_sm = data_high_sm.drop(columns=["target"])
            y_high_sm = data_high_sm["target"]
        except Exception:
            X_high_sm = X_train_pos[top20_reg][high_mask_tr]
            y_high_sm = np.log1p(y_tr_pos[high_mask_tr])

        reg_high = RandomForestRegressor(**RF_PARAMS_REG)
        reg_high.fit(X_high_sm, y_high_sm)

        # Train Prediction
        y_pred_train = np.zeros(len(y_tr_pos))
        y_pred_train[low_mask_tr]  = np.expm1(reg_low.predict(X_train_pos[top20_reg][low_mask_tr]))
        y_pred_train[high_mask_tr] = np.expm1(reg_high.predict(X_train_pos[top20_reg][high_mask_tr]))
        y_pred_train = np.clip(y_pred_train, 0, None)

        # Test Prediction
        y_pred_test = np.zeros(len(y_te_pos))
        if low_pred_te.sum() > 0:
            y_pred_test[low_pred_te]  = np.expm1(reg_low.predict(X_test_pos[top20_reg][low_pred_te]))
        if high_pred_te.sum() > 0:
            y_pred_test[high_pred_te] = np.expm1(reg_high.predict(X_test_pos[top20_reg][high_pred_te]))
        y_pred_test = np.clip(y_pred_test, 0, None)

        # Overall Metrics
        rmse_tr, mae_tr, nmae_tr, r2_tr, nrmse_rng_tr, nrmse_std_tr, cov_tr, agg_tr = calc_metrics(y_tr_pos.values, y_pred_train)
        rmse_te, mae_te, nmae_te, r2_te, nrmse_rng_te, nrmse_std_te, cov_te, agg_te = calc_metrics(y_te_pos.values, y_pred_test)

        # Low / High R2
        r2_low_tr  = r2_score(y_tr_pos[low_mask_tr].values,  y_pred_train[low_mask_tr])
        r2_high_tr = r2_score(y_tr_pos[high_mask_tr].values, y_pred_train[high_mask_tr])
        r2_low_te  = r2_score(y_te_pos.values[low_pred_te],  y_pred_test[low_pred_te])  if low_pred_te.sum()  > 1 else np.nan
        r2_high_te = r2_score(y_te_pos.values[high_pred_te], y_pred_test[high_pred_te]) if high_pred_te.sum() > 1 else np.nan

        all_reg_results.append({
            "Clip":             cp,
            "Threshold":        thr,
            "Low_ratio":        round(low_ratio,            3),
            "Tier_Train":       round(train_tier_acc * 100, 2),
            "Tier_Test":        round(test_tier_acc  * 100, 2),
            "Tier_Gap":         round((train_tier_acc - test_tier_acc) * 100, 2),
            "R2_Train":         round(r2_tr           * 100, 2),
            "MAE_Train":        round(mae_tr,               2),
            "NMAE_Train":       round(nmae_tr,              4),
            "R2_Test":          round(r2_te            * 100, 2),
            "MAE_Test":         round(mae_te,               2),
            "NMAE_Test":        round(nmae_te,              4),
            "R2_Gap":           round((r2_tr - r2_te)  * 100, 2),
            "MAE_Gap":          round(mae_tr - mae_te,      2),
            "NMAE_Gap":         round(nmae_tr - nmae_te,    4),
            "NRMSE_Rng_Train":  round(nrmse_rng_tr,         2),
            "NRMSE_Rng_Test":   round(nrmse_rng_te,         2),
            "NRMSE_Std_Train":  round(nrmse_std_tr,         2),
            "NRMSE_Std_Test":   round(nrmse_std_te,         2),
            "COV_Train":        round(cov_tr           * 100, 2),
            "COV_Test":         round(cov_te           * 100, 2),
            "Err_Train":        round(agg_tr,               2),
            "Err_Test":         round(agg_te,               2),
            "R2_Low_Train":     round(r2_low_tr  * 100, 2),
            "R2_High_Train":    round(r2_high_tr * 100, 2),
            "R2_Low_Test":      round(r2_low_te  * 100, 2) if not np.isnan(r2_low_te)  else np.nan,
            "R2_High_Test":     round(r2_high_te * 100, 2) if not np.isnan(r2_high_te) else np.nan,
        })

        print(f"  thr={thr:5d}  "
              f"R2_Tr={r2_tr*100:.2f}%  R2_Te={r2_te*100:.2f}%  "
              f"R2_Low_Tr={r2_low_tr*100:.2f}%  R2_Low_Te={r2_low_te*100:.2f}%  "
              f"R2_High_Tr={r2_high_tr*100:.2f}%  R2_High_Te={r2_high_te*100:.2f}%  "
              f"COV_Te={cov_te*100:.2f}%  Err_Te={agg_te:.2f}%")

all_reg_df = pd.DataFrame(all_reg_results)

# =========================================================
# Best per Clip (by R2_Test)
# =========================================================
best_per_clip = []

for cp in clip_percentiles:
    subset = all_reg_df[all_reg_df["Clip"] == cp]
    best   = subset.loc[subset["R2_Test"].idxmax()]
    best_per_clip.append({
        "Clip":             cp,
        "Best_Thr":         int(best["Threshold"]),
        "Low_ratio":        best["Low_ratio"],
        "R2_Train":         best["R2_Train"],
        "MAE_Train":        best["MAE_Train"],
        "NMAE_Train":       best["NMAE_Train"],
        "R2_Test":          best["R2_Test"],
        "MAE_Test":         best["MAE_Test"],
        "NMAE_Test":        best["NMAE_Test"],
        "R2_Gap":           best["R2_Gap"],
        "MAE_Gap":          best["MAE_Gap"],
        "NMAE_Gap":         best["NMAE_Gap"],
        "NRMSE_Rng_Train":  best["NRMSE_Rng_Train"],
        "NRMSE_Rng_Test":   best["NRMSE_Rng_Test"],
        "NRMSE_Std_Train":  best["NRMSE_Std_Train"],
        "NRMSE_Std_Test":   best["NRMSE_Std_Test"],
        "COV_Train":        best["COV_Train"],
        "COV_Test":         best["COV_Test"],
        "Err_Train":        best["Err_Train"],
        "Err_Test":         best["Err_Test"],
        "Tier_Train":       best["Tier_Train"],
        "Tier_Test":        best["Tier_Test"],
        "Tier_Gap":         best["Tier_Gap"],
        "R2_Low_Train":     best["R2_Low_Train"],
        "R2_High_Train":    best["R2_High_Train"],
        "R2_Low_Test":      best["R2_Low_Test"],
        "R2_High_Test":     best["R2_High_Test"],
    })

best_df = pd.DataFrame(best_per_clip)

print("\n" + "="*170)
print("=== Best per Clip (by R2_Test) — Independent Regression on y>0 Samples + SMOGN ===")
print("="*170)
print(f"{'Clip':>6} {'Thr':>6} {'Low_r':>7} "
      f"{'R2_Tr':>8} {'MAE_Tr':>9} {'NMAE_Tr':>9} "
      f"{'R2_Te':>8} {'MAE_Te':>9} {'NMAE_Te':>9} "
      f"{'R2_Gap':>8} "
      f"{'COV_Tr':>8} {'COV_Te':>8} "
      f"{'Err_Tr':>8} {'Err_Te':>8} "
      f"{'R2_Low_Te':>10} {'R2_Hi_Te':>10} "
      f"{'Tier_Gap':>10}")
print("-" * 170)
for _, row in best_df.iterrows():
    print(f"{int(row['Clip']):>6} {int(row['Best_Thr']):>6} {row['Low_ratio']:>7.3f} "
          f"{row['R2_Train']:>8.2f} {row['MAE_Train']:>9.2f} {row['NMAE_Train']:>9.4f} "
          f"{row['R2_Test']:>8.2f} {row['MAE_Test']:>9.2f} {row['NMAE_Test']:>9.4f} "
          f"{row['R2_Gap']:>8.2f} "
          f"{row['COV_Train']:>8.2f} {row['COV_Test']:>8.2f} "
          f"{row['Err_Train']:>8.2f} {row['Err_Test']:>8.2f} "
          f"{row['R2_Low_Test']:>10.2f} {row['R2_High_Test']:>10.2f} "
          f"{row['Tier_Gap']:>10.2f}")

# =========================================================
# Visualization
# =========================================================
clips = [str(cp) for cp in clip_percentiles]

fig, axes = plt.subplots(2, 4, figsize=(22, 10))

metrics_plot = [
    ("R2_Train",        "R2_Test",        "R2 Train vs Test (%)"),
    ("MAE_Train",       "MAE_Test",       "MAE Train vs Test"),
    ("NMAE_Train",      "NMAE_Test",      "NMAE Train vs Test"),
    ("NRMSE_Std_Train", "NRMSE_Std_Test", "NRMSE (Std) Train vs Test (%)"),
    ("COV_Train",       "COV_Test",       "COV Train vs Test (%)"),
    ("Err_Train",       "Err_Test",       "%Error_agg Train vs Test (%)"),
    ("R2_Low_Train",    "R2_Low_Test",    "R2 Low Group Train vs Test (%)"),
    ("R2_High_Train",   "R2_High_Test",   "R2 High Group Train vs Test (%)"),
]

for ax, (tr_col, te_col, title) in zip(axes.flatten(), metrics_plot):
    tr_vals = best_df[tr_col].tolist()
    ax.plot(clips, tr_vals, "o-", color="steelblue", label="Train")
    if te_col:
        te_vals = best_df[te_col].tolist()
        ax.plot(clips, te_vals, "o-", color="tomato", label="Test")
        for i, (tr, te) in enumerate(zip(tr_vals, te_vals)):
            ax.annotate(f"{tr:.2f}", (i, tr), textcoords="offset points",
                        xytext=(0,  6), ha="center", fontsize=7, color="steelblue")
            ax.annotate(f"{te:.2f}", (i, te), textcoords="offset points",
                        xytext=(0,-14), ha="center", fontsize=7, color="tomato")
    ax.set_title(title)
    ax.set_xlabel("Clip %")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle("Best per Clip (by R2_Test) — Independent Regression + SMOGN", fontsize=13)
plt.tight_layout()
plt.show()

# =========================================================
# Overall Best
# =========================================================
best_overall = best_df.loc[best_df["R2_Test"].idxmax()]
print(f"\n=== Overall Best (by R2_Test) ===")
print(f"  Clip          : {int(best_overall['Clip'])}%")
print(f"  Threshold     : {int(best_overall['Best_Thr'])}")
print(f"  R2_Train      : {best_overall['R2_Train']}%")
print(f"  R2_Test       : {best_overall['R2_Test']}%")
print(f"  R2_Gap        : {best_overall['R2_Gap']}%")
print(f"  R2_Low_Train  : {best_overall['R2_Low_Train']}%")
print(f"  R2_Low_Test   : {best_overall['R2_Low_Test']}%")
print(f"  R2_High_Train : {best_overall['R2_High_Train']}%")
print(f"  R2_High_Test  : {best_overall['R2_High_Test']}%")
print(f"  MAE_Test      : {best_overall['MAE_Test']}")
print(f"  COV_Test      : {best_overall['COV_Test']}%")
print(f"  Err_Test      : {best_overall['Err_Test']}%")
print(f"  Tier_Gap      : {best_overall['Tier_Gap']}%")

# =========================================================
# Actual vs Predicted Distribution (Overall Best)
# =========================================================
cp_best  = int(best_overall["Clip"])
thr_best = int(best_overall["Best_Thr"])

X_test_pos  = clip_results_pos[cp_best]["X_test"]
y_te_pos    = clip_results_pos[cp_best]["y_test_reg"]
y_tr_pos    = clip_results_pos[cp_best]["y_train_reg"]
X_train_pos = clip_results_pos[cp_best]["X_train"]

low_mask_tr  = y_tr_pos <= thr_best
high_mask_tr = y_tr_pos >  thr_best

tier_label_tr = pd.Series(
    np.where(y_tr_pos <= thr_best, "low", "high"),
    index=y_tr_pos.index
)
try:
    smote_l2 = SMOTE(random_state=42)
    X_sm_l2, y_sm_l2 = smote_l2.fit_resample(X_train_pos[top20_reg], tier_label_tr)
except Exception:
    X_sm_l2 = X_train_pos[top20_reg]
    y_sm_l2 = tier_label_tr

clf_tier = RandomForestClassifier(**RF_PARAMS_REG)
clf_tier.fit(X_sm_l2, y_sm_l2)
tier_pred    = clf_tier.predict(X_test_pos[top20_reg])
low_pred_te  = tier_pred == "low"
high_pred_te = tier_pred == "high"

# SMOGN for final model
try:
    data_low = X_train_pos[top20_reg][low_mask_tr].copy().reset_index(drop=True)
    data_low["target"] = np.log1p(y_tr_pos[low_mask_tr].values)
    data_low_sm = smogn.smoter(data=data_low, y="target", k=min(5, len(data_low)-1), samp_method="balance", seed=42)
    X_low_sm = data_low_sm.drop(columns=["target"])
    y_low_sm = data_low_sm["target"]
except Exception:
    X_low_sm = X_train_pos[top20_reg][low_mask_tr]
    y_low_sm = np.log1p(y_tr_pos[low_mask_tr])

try:
    data_high = X_train_pos[top20_reg][high_mask_tr].copy().reset_index(drop=True)
    data_high["target"] = np.log1p(y_tr_pos[high_mask_tr].values)
    data_high_sm = smogn.smoter(data=data_high, y="target", k=min(5, len(data_high)-1), samp_method="balance", seed=42)
    X_high_sm = data_high_sm.drop(columns=["target"])
    y_high_sm = data_high_sm["target"]
except Exception:
    X_high_sm = X_train_pos[top20_reg][high_mask_tr]
    y_high_sm = np.log1p(y_tr_pos[high_mask_tr])

reg_low = RandomForestRegressor(**RF_PARAMS_REG)
reg_low.fit(X_low_sm, y_low_sm)

reg_high = RandomForestRegressor(**RF_PARAMS_REG)
reg_high.fit(X_high_sm, y_high_sm)

y_pred_test = np.zeros(len(y_te_pos))
if low_pred_te.sum() > 0:
    y_pred_test[low_pred_te]  = np.expm1(reg_low.predict(X_test_pos[top20_reg][low_pred_te]))
if high_pred_te.sum() > 0:
    y_pred_test[high_pred_te] = np.expm1(reg_high.predict(X_test_pos[top20_reg][high_pred_te]))
y_pred_test = np.clip(y_pred_test, 0, None)

# Metrics
r2_overall  = r2_score(y_te_pos.values, y_pred_test)
mae_overall = np.mean(np.abs(y_te_pos.values - y_pred_test))
r2_low   = r2_score(y_te_pos.values[low_pred_te],  y_pred_test[low_pred_te])  if low_pred_te.sum()  > 1 else np.nan
mae_low  = np.mean(np.abs(y_te_pos.values[low_pred_te]  - y_pred_test[low_pred_te]))  if low_pred_te.sum()  > 1 else np.nan
r2_high  = r2_score(y_te_pos.values[high_pred_te], y_pred_test[high_pred_te]) if high_pred_te.sum() > 1 else np.nan
mae_high = np.mean(np.abs(y_te_pos.values[high_pred_te] - y_pred_test[high_pred_te])) if high_pred_te.sum() > 1 else np.nan

x_min = 0
x_max = max(y_te_pos.values.max(), y_pred_test.max()) * 1.05
bins  = np.linspace(x_min, x_max, 50)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(y_te_pos.values, bins=bins, alpha=0.6, color="steelblue", label="Actual")
axes[0].hist(y_pred_test,     bins=bins, alpha=0.6, color="tomato",    label="Predicted")
axes[0].set_xlim(x_min, x_max)
axes[0].set_title(f"Overall  (Clip={cp_best}%  Thr={thr_best})\nR2={r2_overall*100:.2f}%  MAE={mae_overall:.2f}")
axes[0].set_xlabel("VolBoth_sum")
axes[0].set_ylabel("Count")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].hist(y_te_pos.values[low_pred_te], bins=bins, alpha=0.6, color="steelblue", label="Actual")
axes[1].hist(y_pred_test[low_pred_te],     bins=bins, alpha=0.6, color="tomato",    label="Predicted")
axes[1].set_xlim(x_min, x_max)
axes[1].set_title(f"Low Group (y ≤ {thr_best})\nR2={r2_low*100:.2f}%  MAE={mae_low:.2f}")
axes[1].set_xlabel("VolBoth_sum")
axes[1].legend()
axes[1].grid(alpha=0.3)

axes[2].hist(y_te_pos.values[high_pred_te], bins=bins, alpha=0.6, color="steelblue", label="Actual")
axes[2].hist(y_pred_test[high_pred_te],     bins=bins, alpha=0.6, color="tomato",    label="Predicted")
axes[2].set_xlim(x_min, x_max)
axes[2].set_title(f"High Group (y > {thr_best})\nR2={r2_high*100:.2f}%  MAE={mae_high:.2f}")
axes[2].set_xlabel("VolBoth_sum")
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.suptitle(f"Actual vs Predicted Distribution  Clip={cp_best}%  Thr={thr_best} + SMOGN", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# =========================================================
# Actual vs Predicted Distribution (Overall Best)
# =========================================================
cp_best  = int(best_overall["Clip"])
thr_best = int(best_overall["Best_Thr"])
X_test_pos  = clip_results_pos[cp_best]["X_test"]
y_te_pos    = clip_results_pos[cp_best]["y_test_reg"]
y_tr_pos    = clip_results_pos[cp_best]["y_train_reg"]
X_train_pos = clip_results_pos[cp_best]["X_train"]

# Retrain best model
low_mask_tr  = y_tr_pos <= thr_best
high_mask_tr = y_tr_pos >  thr_best
tier_label_tr = pd.Series(
    np.where(y_tr_pos <= thr_best, "low", "high"),
    index=y_tr_pos.index
)
try:
    smote_l2 = SMOTE(random_state=42)
    X_sm_l2, y_sm_l2 = smote_l2.fit_resample(X_train_pos[top20_reg], tier_label_tr)
except Exception:
    X_sm_l2 = X_train_pos[top20_reg]
    y_sm_l2 = tier_label_tr

clf_tier = RandomForestClassifier(**RF_PARAMS_REG)
clf_tier.fit(X_sm_l2, y_sm_l2)
tier_pred    = clf_tier.predict(X_test_pos[top20_reg])
low_pred_te  = tier_pred == "low"
high_pred_te = tier_pred == "high"

reg_low = RandomForestRegressor(**RF_PARAMS_REG)
reg_low.fit(X_train_pos[top20_reg][low_mask_tr], np.log1p(y_tr_pos[low_mask_tr]))
reg_high = RandomForestRegressor(**RF_PARAMS_REG)
reg_high.fit(X_train_pos[top20_reg][high_mask_tr], np.log1p(y_tr_pos[high_mask_tr]))

y_pred_test = np.zeros(len(y_te_pos))
if low_pred_te.sum() > 0:
    y_pred_test[low_pred_te]  = np.expm1(reg_low.predict(X_test_pos[top20_reg][low_pred_te]))
if high_pred_te.sum() > 0:
    y_pred_test[high_pred_te] = np.expm1(reg_high.predict(X_test_pos[top20_reg][high_pred_te]))
y_pred_test = np.clip(y_pred_test, 0, None)

# Metrics
r2_overall  = r2_score(y_te_pos.values, y_pred_test)
mae_overall = np.mean(np.abs(y_te_pos.values - y_pred_test))
r2_low   = r2_score(y_te_pos.values[low_pred_te],  y_pred_test[low_pred_te])  if low_pred_te.sum()  > 1 else np.nan
mae_low  = np.mean(np.abs(y_te_pos.values[low_pred_te]  - y_pred_test[low_pred_te]))  if low_pred_te.sum()  > 1 else np.nan
r2_high  = r2_score(y_te_pos.values[high_pred_te], y_pred_test[high_pred_te]) if high_pred_te.sum() > 1 else np.nan
mae_high = np.mean(np.abs(y_te_pos.values[high_pred_te] - y_pred_test[high_pred_te])) if high_pred_te.sum() > 1 else np.nan

def fmt(v):
    return f"{v:.2f}" if v is not None and not np.isnan(v) else "nan"

# Shared x-axis range
x_min = 0
x_max = max(y_te_pos.values.max(), y_pred_test.max()) * 1.05
bins  = np.linspace(x_min, x_max, 50)

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Overall
axes[0].hist(y_te_pos.values, bins=bins, alpha=0.6, color="steelblue", label="Actual")
axes[0].hist(y_pred_test,     bins=bins, alpha=0.6, color="tomato",    label="Predicted")
axes[0].set_xlim(x_min, x_max)
axes[0].set_title(f"Overall  (Clip={cp_best}%  Thr={thr_best})\nR2={r2_overall*100:.2f}%  MAE={mae_overall:.2f}")
axes[0].set_xlabel("VolBoth_sum")
axes[0].set_ylabel("Count")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Low group
axes[1].hist(y_te_pos.values[low_pred_te], bins=bins, alpha=0.6, color="steelblue", label="Actual")
axes[1].hist(y_pred_test[low_pred_te],     bins=bins, alpha=0.6, color="tomato",    label="Predicted")
axes[1].set_xlim(x_min, x_max)
axes[1].set_title(f"Low Group (y <= {thr_best})\nR2={fmt(r2_low*100 if not np.isnan(r2_low) else np.nan)}%  MAE={fmt(mae_low)}")
axes[1].set_xlabel("VolBoth_sum")
axes[1].legend()
axes[1].grid(alpha=0.3)

# High group
axes[2].hist(y_te_pos.values[high_pred_te], bins=bins, alpha=0.6, color="steelblue", label="Actual")
axes[2].hist(y_pred_test[high_pred_te],     bins=bins, alpha=0.6, color="tomato",    label="Predicted")
axes[2].set_xlim(x_min, x_max)
axes[2].set_title(f"High Group (y > {thr_best})\nR2={fmt(r2_high*100 if not np.isnan(r2_high) else np.nan)}%  MAE={fmt(mae_high)}")
axes[2].set_xlabel("VolBoth_sum")
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.suptitle(f"Actual vs Predicted Distribution  Clip={cp_best}%  Thr={thr_best}", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# =========================================================
# y Volume Distribution
# =========================================================
import matplotlib.pyplot as plt
import numpy as np

y_pos = y_reg_raw[y_reg_raw > 0]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histogram
axes[0].hist(y_pos, bins=50, color="steelblue", edgecolor="white")
axes[0].set_title("y>0 Distribution")
axes[0].set_xlabel("VolBoth_sum")
axes[0].set_ylabel("Count")
axes[0].grid(alpha=0.3)

# Log-scale Histogram
axes[1].hist(y_pos, bins=50, color="steelblue", edgecolor="white")
axes[1].set_yscale("log")
axes[1].set_title("y>0 Distribution (Log Scale)")
axes[1].set_xlabel("VolBoth_sum")
axes[1].set_ylabel("Count (log)")
axes[1].grid(alpha=0.3)

# Boxplot
axes[2].boxplot(y_pos, vert=True)
axes[2].set_title("y>0 Boxplot")
axes[2].set_ylabel("VolBoth_sum")
axes[2].grid(alpha=0.3)

plt.suptitle("y>0 Distribution", fontsize=13)
plt.tight_layout()
plt.show()

# Percentile summary
print("=== Percentile ===")
for p in [50, 60, 70, 75, 80, 85, 90, 95, 99, 100]:
    print(f"  {p:>3}%: {y_pos.quantile(p/100):.1f}")

In [ ]:
# =========================================================
# Cell 2-B (Fixed Clip): Layer 2 — Regression (Independent Model)
# Fixed y_upper=2700, samples > 2700 excluded
# Threshold Sweep only
# =========================================================

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

def calc_metrics(y_true, y_pred):
    rmse       = np.sqrt(mean_squared_error(y_true, y_pred))
    mae        = np.mean(np.abs(y_true - y_pred))
    nmae       = mae / (y_true.max() - y_true.min())
    r2         = r2_score(y_true, y_pred)
    nrmse_rng  = rmse / (y_true.max() - y_true.min()) * 100
    nrmse_std  = rmse / y_true.std() * 100
    cov        = rmse / y_true.mean()
    agg        = np.abs(y_true - y_pred).sum() / y_true.sum() * 100
    return rmse, mae, nmae, r2, nrmse_rng, nrmse_std, cov, agg

def fmt(v):
    return f"{v*100:.2f}" if v is not None and not np.isnan(v) else "nan"

RF_PARAMS_REG = dict(
    n_estimators     = 50,
    max_depth        = None,
    min_samples_leaf = 1,
    max_features     = 7,
    random_state     = 42,
    n_jobs           = -1
)

# =========================================================
# CONFIG
# =========================================================
FIXED_Y_UPPER   = 2700
thr_candidates  = list(range(100, FIXED_Y_UPPER, 25))
all_reg_results = []

# =========================================================
# Common Columns
# =========================================================
bldg_cols    = [c for c in df_model.columns if c.startswith("bldg")]
occtype_cols = [c for c in df_model.columns if c.startswith("occtype")]
grouped_cols = set(bldg_cols + occtype_cols)

# =========================================================
# Use only y>0 AND y<=2700 samples
# =========================================================
pos_mask   = (y_reg_raw > 0) & (y_reg_raw <= FIXED_Y_UPPER)
df_pos     = df_model[pos_mask]
y_pos_raw2 = y_reg_raw[pos_mask]

print(f"Total y>0 samples: {(y_reg_raw > 0).sum()}")
print(f"y>0 & y<={FIXED_Y_UPPER} samples: {len(df_pos)}")
print(f"Excluded samples: {(y_reg_raw > FIXED_Y_UPPER).sum()}")
print(f"\ny value distribution:")
print(y_pos_raw2.describe())

median_val = int(y_pos_raw2.median())
print(f"\nMedian: {median_val}")

y_stratify_pos = y_pos_raw2.apply(
    lambda x: "low" if x <= median_val else "high"
)

X_train_pos, X_test_pos, _, _ = train_test_split(
    df_pos, y_stratify_pos,
    test_size=0.15,
    random_state=42,
    stratify=y_stratify_pos
)

y_train_pos = y_pos_raw2.loc[X_train_pos.index]
y_test_pos  = y_pos_raw2.loc[X_test_pos.index]

print(f"\nTrain: {len(X_train_pos)}  Test: {len(X_test_pos)}")
print(f"Train y max: {y_train_pos.max():.1f}")
print(f"Test  y max: {y_test_pos.max():.1f}")

# =========================================================
# Top 20 Features
# =========================================================
reg_feat_tmp = RandomForestRegressor(**RF_PARAMS_REG)
reg_feat_tmp.fit(X_train_pos, np.log1p(y_train_pos))

fi_reg_raw = pd.DataFrame({
    "Feature":    X_train_pos.columns,
    "Importance": reg_feat_tmp.feature_importances_
}).sort_values("Importance", ascending=False)

grouped_imp_reg = {}
grouped_imp_reg["bldg_type"] = fi_reg_raw[fi_reg_raw["Feature"].isin(bldg_cols)]["Importance"].sum()
grouped_imp_reg["occtype"]   = fi_reg_raw[fi_reg_raw["Feature"].isin(occtype_cols)]["Importance"].sum()
for _, row in fi_reg_raw.iterrows():
    if row["Feature"] not in grouped_cols:
        grouped_imp_reg[row["Feature"]] = row["Importance"]

grouped_imp_reg_df = pd.DataFrame(
    grouped_imp_reg.items(), columns=["feature", "importance"]
).sort_values("importance", ascending=False).reset_index(drop=True)

top20_reg_groups = grouped_imp_reg_df.head(20)["feature"].tolist()
top20_reg        = expand_features(top20_reg_groups, bldg_cols, occtype_cols)

print("\n--- Top 20 Features ---")
print(f"{'Rank':>4}  {'Feature':30} {'Importance':>12}")
print("-" * 50)
for idx, row in grouped_imp_reg_df.iterrows():
    tag = "* TOP20" if idx < 20 else ""
    print(f"{idx+1:>4}  {row['feature']:30} {row['importance']:>12.6f}  {tag}")
print(f"\nTop 20 groups -> expanded to {len(top20_reg)} columns")

# =========================================================
# Threshold Sweep
# =========================================================
print(f"\nThreshold candidates: {len(thr_candidates)}  range: {thr_candidates[0]}~{thr_candidates[-1]}")
print(f"\n{'='*60}")
print(f"Fixed y_upper={FIXED_Y_UPPER} (samples > {FIXED_Y_UPPER} excluded)")
print(f"{'='*60}")
print(f"Train pos: {len(X_train_pos)}  Test pos: {len(X_test_pos)}")

for thr in thr_candidates:

    low_mask_tr  = y_train_pos <= thr
    high_mask_tr = y_train_pos >  thr

    n_low_tr  = low_mask_tr.sum()
    n_high_tr = high_mask_tr.sum()

    if n_low_tr < 5 or n_high_tr < 5:
        continue

    low_ratio = n_low_tr / (n_low_tr + n_high_tr)
    if not (0.30 <= low_ratio <= 0.70):
        continue

    # Tier Classifier (SMOTE)
    tier_label_tr = pd.Series(
        np.where(y_train_pos <= thr, "low", "high"),
        index=y_train_pos.index
    )
    try:
        smote_l2 = SMOTE(random_state=42)
        X_sm_l2, y_sm_l2 = smote_l2.fit_resample(
            X_train_pos[top20_reg], tier_label_tr
        )
    except Exception:
        X_sm_l2 = X_train_pos[top20_reg]
        y_sm_l2 = tier_label_tr

    clf_tier = RandomForestClassifier(**RF_PARAMS_REG)
    clf_tier.fit(X_sm_l2, y_sm_l2)

    train_tier_acc = (clf_tier.predict(X_sm_l2) == y_sm_l2).mean()
    tier_pred      = clf_tier.predict(X_test_pos[top20_reg])
    tier_true      = np.where(y_test_pos <= thr, "low", "high")
    test_tier_acc  = (tier_pred == tier_true).mean()

    low_pred_te  = tier_pred == "low"
    high_pred_te = tier_pred == "high"

    reg_low = RandomForestRegressor(**RF_PARAMS_REG)
    reg_low.fit(
        X_train_pos[top20_reg][low_mask_tr],
        np.log1p(y_train_pos[low_mask_tr])
    )

    reg_high = RandomForestRegressor(**RF_PARAMS_REG)
    reg_high.fit(
        X_train_pos[top20_reg][high_mask_tr],
        np.log1p(y_train_pos[high_mask_tr])
    )

    # Train Prediction
    y_pred_train = np.zeros(len(y_train_pos))
    y_pred_train[low_mask_tr]  = np.expm1(reg_low.predict(X_train_pos[top20_reg][low_mask_tr]))
    y_pred_train[high_mask_tr] = np.expm1(reg_high.predict(X_train_pos[top20_reg][high_mask_tr]))
    y_pred_train = np.clip(y_pred_train, 0, None)

    # Test Prediction
    y_pred_test = np.zeros(len(y_test_pos))
    if low_pred_te.sum() > 0:
        y_pred_test[low_pred_te]  = np.expm1(reg_low.predict(X_test_pos[top20_reg][low_pred_te]))
    if high_pred_te.sum() > 0:
        y_pred_test[high_pred_te] = np.expm1(reg_high.predict(X_test_pos[top20_reg][high_pred_te]))
    y_pred_test = np.clip(y_pred_test, 0, None)

    # Metrics
    rmse_tr, mae_tr, nmae_tr, r2_tr, nrmse_rng_tr, nrmse_std_tr, cov_tr, agg_tr = calc_metrics(y_train_pos.values, y_pred_train)
    rmse_te, mae_te, nmae_te, r2_te, nrmse_rng_te, nrmse_std_te, cov_te, agg_te = calc_metrics(y_test_pos.values, y_pred_test)

    r2_low_tr  = r2_score(y_train_pos[low_mask_tr].values,  y_pred_train[low_mask_tr])
    r2_high_tr = r2_score(y_train_pos[high_mask_tr].values, y_pred_train[high_mask_tr])
    r2_low_te  = r2_score(y_test_pos.values[low_pred_te],  y_pred_test[low_pred_te])  if low_pred_te.sum()  > 1 else np.nan
    r2_high_te = r2_score(y_test_pos.values[high_pred_te], y_pred_test[high_pred_te]) if high_pred_te.sum() > 1 else np.nan

    all_reg_results.append({
        "Threshold":        thr,
        "Low_ratio":        round(low_ratio,            3),
        "Tier_Train":       round(train_tier_acc * 100, 2),
        "Tier_Test":        round(test_tier_acc  * 100, 2),
        "Tier_Gap":         round((train_tier_acc - test_tier_acc) * 100, 2),
        "R2_Train":         round(r2_tr           * 100, 2),
        "MAE_Train":        round(mae_tr,               2),
        "NMAE_Train":       round(nmae_tr,              4),
        "R2_Test":          round(r2_te            * 100, 2),
        "MAE_Test":         round(mae_te,               2),
        "NMAE_Test":        round(nmae_te,              4),
        "R2_Gap":           round((r2_tr - r2_te)  * 100, 2),
        "MAE_Gap":          round(mae_tr - mae_te,      2),
        "NMAE_Gap":         round(nmae_tr - nmae_te,    4),
        "NRMSE_Rng_Train":  round(nrmse_rng_tr,         2),
        "NRMSE_Rng_Test":   round(nrmse_rng_te,         2),
        "NRMSE_Std_Train":  round(nrmse_std_tr,         2),
        "NRMSE_Std_Test":   round(nrmse_std_te,         2),
        "COV_Train":        round(cov_tr           * 100, 2),
        "COV_Test":         round(cov_te           * 100, 2),
        "Err_Train":        round(agg_tr,               2),
        "Err_Test":         round(agg_te,               2),
        "R2_Low_Train":     round(r2_low_tr  * 100, 2),
        "R2_High_Train":    round(r2_high_tr * 100, 2),
        "R2_Low_Test":      round(r2_low_te  * 100, 2) if not np.isnan(r2_low_te)  else np.nan,
        "R2_High_Test":     round(r2_high_te * 100, 2) if not np.isnan(r2_high_te) else np.nan,
    })

    # FIX: safe formatting for nan values
    print(f"  thr={thr:5d}  "
          f"R2_Tr={r2_tr*100:.2f}%  R2_Te={r2_te*100:.2f}%  "
          f"R2_Low_Te={fmt(r2_low_te)}%  R2_High_Te={fmt(r2_high_te)}%  "
          f"COV_Te={cov_te*100:.2f}%  Err_Te={agg_te:.2f}%")

all_reg_df = pd.DataFrame(all_reg_results)

# =========================================================
# Best (by R2_Test)
# =========================================================
best = all_reg_df.loc[all_reg_df["R2_Test"].idxmax()]

print("\n" + "="*150)
print(f"=== Best Result (by R2_Test) — Fixed y_upper={FIXED_Y_UPPER} (excluded > {FIXED_Y_UPPER}) ===")
print("="*150)
print(f"{'Thr':>6} {'Low_r':>7} "
      f"{'R2_Tr':>8} {'MAE_Tr':>9} {'NMAE_Tr':>9} "
      f"{'R2_Te':>8} {'MAE_Te':>9} {'NMAE_Te':>9} "
      f"{'R2_Gap':>8} "
      f"{'COV_Tr':>8} {'COV_Te':>8} "
      f"{'Err_Tr':>8} {'Err_Te':>8} "
      f"{'R2_Low_Te':>10} {'R2_Hi_Te':>10} "
      f"{'Tier_Gap':>10}")
print("-" * 150)
print(f"{int(best['Threshold']):>6} {best['Low_ratio']:>7.3f} "
      f"{best['R2_Train']:>8.2f} {best['MAE_Train']:>9.2f} {best['NMAE_Train']:>9.4f} "
      f"{best['R2_Test']:>8.2f} {best['MAE_Test']:>9.2f} {best['NMAE_Test']:>9.4f} "
      f"{best['R2_Gap']:>8.2f} "
      f"{best['COV_Train']:>8.2f} {best['COV_Test']:>8.2f} "
      f"{best['Err_Train']:>8.2f} {best['Err_Test']:>8.2f} "
      f"{best['R2_Low_Test']:>10.2f} {best['R2_High_Test']:>10.2f} "
      f"{best['Tier_Gap']:>10.2f}")

# =========================================================
# Visualization — Threshold Sweep
# =========================================================
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

thresholds = all_reg_df["Threshold"].tolist()

axes[0,0].plot(thresholds, all_reg_df["R2_Train"], "o-", color="steelblue", label="Train", markersize=3)
axes[0,0].plot(thresholds, all_reg_df["R2_Test"],  "o-", color="tomato",    label="Test",  markersize=3)
axes[0,0].axvline(int(best["Threshold"]), color="green", linestyle="--", lw=1.5, label=f"Best={int(best['Threshold'])}")
axes[0,0].set_title("R2 Train vs Test (%)")
axes[0,0].set_xlabel("Threshold")
axes[0,0].legend(fontsize=8)
axes[0,0].grid(alpha=0.3)

axes[0,1].plot(thresholds, all_reg_df["R2_Low_Test"],  "o-", color="steelblue", label="Low",  markersize=3)
axes[0,1].plot(thresholds, all_reg_df["R2_High_Test"], "o-", color="tomato",    label="High", markersize=3)
axes[0,1].axvline(int(best["Threshold"]), color="green", linestyle="--", lw=1.5)
axes[0,1].set_title("R2 Low vs High Test (%)")
axes[0,1].set_xlabel("Threshold")
axes[0,1].legend(fontsize=8)
axes[0,1].grid(alpha=0.3)

axes[0,2].plot(thresholds, all_reg_df["MAE_Train"], "o-", color="steelblue", label="Train", markersize=3)
axes[0,2].plot(thresholds, all_reg_df["MAE_Test"],  "o-", color="tomato",    label="Test",  markersize=3)
axes[0,2].axvline(int(best["Threshold"]), color="green", linestyle="--", lw=1.5)
axes[0,2].set_title("MAE Train vs Test")
axes[0,2].set_xlabel("Threshold")
axes[0,2].legend(fontsize=8)
axes[0,2].grid(alpha=0.3)

axes[1,0].plot(thresholds, all_reg_df["COV_Train"], "o-", color="steelblue", label="Train", markersize=3)
axes[1,0].plot(thresholds, all_reg_df["COV_Test"],  "o-", color="tomato",    label="Test",  markersize=3)
axes[1,0].axvline(int(best["Threshold"]), color="green", linestyle="--", lw=1.5)
axes[1,0].set_title("COV Train vs Test (%)")
axes[1,0].set_xlabel("Threshold")
axes[1,0].legend(fontsize=8)
axes[1,0].grid(alpha=0.3)

axes[1,1].plot(thresholds, all_reg_df["Err_Train"], "o-", color="steelblue", label="Train", markersize=3)
axes[1,1].plot(thresholds, all_reg_df["Err_Test"],  "o-", color="tomato",    label="Test",  markersize=3)
axes[1,1].axvline(int(best["Threshold"]), color="green", linestyle="--", lw=1.5)
axes[1,1].set_title("%Error_agg Train vs Test (%)")
axes[1,1].set_xlabel("Threshold")
axes[1,1].legend(fontsize=8)
axes[1,1].grid(alpha=0.3)

axes[1,2].plot(thresholds, all_reg_df["R2_Gap"], "o-", color="purple", markersize=3)
axes[1,2].axvline(int(best["Threshold"]), color="green", linestyle="--", lw=1.5)
axes[1,2].set_title("R2 Gap (%)")
axes[1,2].set_xlabel("Threshold")
axes[1,2].grid(alpha=0.3)

plt.suptitle(f"Threshold Sweep — Fixed y_upper={FIXED_Y_UPPER} (excluded > {FIXED_Y_UPPER})", fontsize=13)
plt.tight_layout()
plt.show()

# =========================================================
# Actual vs Predicted Distribution (Best)
# =========================================================
thr_best = int(best["Threshold"])

low_mask_tr  = y_train_pos <= thr_best
high_mask_tr = y_train_pos >  thr_best

tier_label_tr = pd.Series(
    np.where(y_train_pos <= thr_best, "low", "high"),
    index=y_train_pos.index
)
try:
    smote_l2 = SMOTE(random_state=42)
    X_sm_l2, y_sm_l2 = smote_l2.fit_resample(X_train_pos[top20_reg], tier_label_tr)
except Exception:
    X_sm_l2 = X_train_pos[top20_reg]
    y_sm_l2 = tier_label_tr

clf_tier = RandomForestClassifier(**RF_PARAMS_REG)
clf_tier.fit(X_sm_l2, y_sm_l2)
tier_pred    = clf_tier.predict(X_test_pos[top20_reg])
low_pred_te  = tier_pred == "low"
high_pred_te = tier_pred == "high"

reg_low = RandomForestRegressor(**RF_PARAMS_REG)
reg_low.fit(X_train_pos[top20_reg][low_mask_tr], np.log1p(y_train_pos[low_mask_tr]))

reg_high = RandomForestRegressor(**RF_PARAMS_REG)
reg_high.fit(X_train_pos[top20_reg][high_mask_tr], np.log1p(y_train_pos[high_mask_tr]))

y_pred_test = np.zeros(len(y_test_pos))
if low_pred_te.sum() > 0:
    y_pred_test[low_pred_te]  = np.expm1(reg_low.predict(X_test_pos[top20_reg][low_pred_te]))
if high_pred_te.sum() > 0:
    y_pred_test[high_pred_te] = np.expm1(reg_high.predict(X_test_pos[top20_reg][high_pred_te]))
y_pred_test = np.clip(y_pred_test, 0, None)

r2_overall  = r2_score(y_test_pos.values, y_pred_test)
mae_overall = np.mean(np.abs(y_test_pos.values - y_pred_test))
r2_low   = r2_score(y_test_pos.values[low_pred_te],  y_pred_test[low_pred_te])  if low_pred_te.sum()  > 1 else np.nan
mae_low  = np.mean(np.abs(y_test_pos.values[low_pred_te]  - y_pred_test[low_pred_te]))  if low_pred_te.sum()  > 1 else np.nan
r2_high  = r2_score(y_test_pos.values[high_pred_te], y_pred_test[high_pred_te]) if high_pred_te.sum() > 1 else np.nan
mae_high = np.mean(np.abs(y_test_pos.values[high_pred_te] - y_pred_test[high_pred_te])) if high_pred_te.sum() > 1 else np.nan

x_min = 0
x_max = max(y_test_pos.values.max(), y_pred_test.max()) * 1.05
bins  = np.linspace(x_min, x_max, 50)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(y_test_pos.values, bins=bins, alpha=0.6, color="steelblue", label="Actual")
axes[0].hist(y_pred_test,       bins=bins, alpha=0.6, color="tomato",    label="Predicted")
axes[0].set_xlim(x_min, x_max)
axes[0].set_title(f"Overall  (y_upper={FIXED_Y_UPPER}  Thr={thr_best})\nR2={r2_overall*100:.2f}%  MAE={mae_overall:.2f}")
axes[0].set_xlabel("VolBoth_sum")
axes[0].set_ylabel("Count")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].hist(y_test_pos.values[low_pred_te], bins=bins, alpha=0.6, color="steelblue", label="Actual")
axes[1].hist(y_pred_test[low_pred_te],       bins=bins, alpha=0.6, color="tomato",    label="Predicted")
axes[1].set_xlim(x_min, x_max)
mae_low_str  = f"{mae_low:.2f}"  if mae_low  is not None and not np.isnan(mae_low)  else "nan"
mae_high_str = f"{mae_high:.2f}" if mae_high is not None and not np.isnan(mae_high) else "nan"

axes[1].set_title(f"Low Group (y <= {thr_best})\nR2={fmt(r2_low)}%  MAE={mae_low_str}")
axes[1].set_xlabel("VolBoth_sum")
axes[1].legend()
axes[1].grid(alpha=0.3)

axes[2].hist(y_test_pos.values[high_pred_te], bins=bins, alpha=0.6, color="steelblue", label="Actual")
axes[2].hist(y_pred_test[high_pred_te],        bins=bins, alpha=0.6, color="tomato",    label="Predicted")
axes[2].set_xlim(x_min, x_max)
axes[2].set_title(f"High Group (y > {thr_best})\nR2={fmt(r2_high)}%  MAE={mae_high_str}")
axes[2].set_xlabel("VolBoth_sum")
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.suptitle(f"Actual vs Predicted Distribution  y_upper={FIXED_Y_UPPER}  Thr={thr_best}", fontsize=13)
plt.tight_layout()
plt.show()

print(f"\n=== Overall Best ===")
print(f"  y_upper      : {FIXED_Y_UPPER}")
print(f"  Excluded     : {(y_reg_raw > FIXED_Y_UPPER).sum()} samples")
print(f"  Threshold    : {thr_best}")
print(f"  R2_Train     : {best['R2_Train']}%")
print(f"  R2_Test      : {best['R2_Test']}%")
print(f"  R2_Gap       : {best['R2_Gap']}%")
print(f"  R2_Low_Test  : {best['R2_Low_Test']}%")
print(f"  R2_High_Test : {best['R2_High_Test']}%")
print(f"  MAE_Test     : {best['MAE_Test']}")
print(f"  COV_Test     : {best['COV_Test']}%")
print(f"  Err_Test     : {best['Err_Test']}%")
print(f"  Tier_Gap     : {best['Tier_Gap']}%")